# 01 — Workload Overview

This notebook summarises what types of computation research groups are running, which codes they use, how many jobs they submit, and how groups are distributed across departments.

**Run `00_setup.ipynb` first** to install packages and authenticate.

In [ ]:
# Download shared analysis module from GitHub
!wget -q https://raw.githubusercontent.com/svaradh/hpc-questionnaire/main/analysis/sheets_client.py
SPREADSHEET_ID = 'PASTE_YOUR_SPREADSHEET_ID_HERE'  # ← change this

In [ ]:
from sheets_client import (
    load_sheets, summarise_sheets, explode_semicolons,
    split_semicolons, map_range_labels,
    CPU_HOURS_LABELS, WALL_TIME_LABELS, MEMORY_LABELS,
    MEMORY_PER_CORE_LABELS, CORES_LABELS, GPU_MEMORY_LABELS,
    JOB_COUNT_LABELS, CPU_HOURS_MIDPOINTS, JOB_COUNT_MIDPOINTS
)
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid')

dfs = load_sheets(SPREADSHEET_ID)
print('Data loaded. Submissions:', len(dfs.get('Submissions', pd.DataFrame())))

---
## Chart 1 — Workload Category Distribution

Shows how common each computation type is across all codes submitted. A code can belong to multiple categories (e.g. a code may be both molecular dynamics and machine learning). Each category is counted separately.

**What to look for:** Which computation types dominate the user base? Are there categories with very few entries that may need special consideration?

In [ ]:
CATEGORY_LABELS = {
    'electronic_structure': 'Electronic structure',
    'molecular_dynamics': 'Molecular dynamics',
    'atomistic_simulation': 'Atomistic simulation',
    'climate_environmental': 'Climate / environmental',
    'cfd': 'CFD',
    'bioinformatics_genomics': 'Bioinformatics / genomics',
    'machine_learning': 'Machine learning / AI',
    'numerical_modelling': 'Numerical modelling',
    'many_body_exact_diagonalisation': 'Many-body / exact diag.',
    'data_analysis': 'Data analysis',
    'other': 'Other',
}

if workloads.empty or 'categories' not in workloads.columns:
    print("No workload category data available.")
else:
    cats = explode_semicolons(workloads, 'categories')
    cats = cats.map(lambda x: CATEGORY_LABELS.get(x, x))
    counts = cats.value_counts()

    fig, ax = plt.subplots(figsize=(7, 7))
    wedges, texts, autotexts = ax.pie(
        counts.values,
        labels=counts.index,
        autopct=lambda p: f'{p:.1f}%\n({int(round(p * counts.sum() / 100))})',
        startangle=140,
        pctdistance=0.75,
        textprops={'fontsize': 9},
    )
    ax.set_title('Workload Category Distribution\n(all codes, all groups)', fontsize=13, pad=16)
    plt.tight_layout()
    plt.show()
    print(counts.to_string())

---
## Chart 2 — Top 15 Codes Used

Shows the most frequently mentioned application and code names across all submissions. Code names are normalised to lowercase for counting.

**What to look for:** Which codes are most widely used? These are candidates for facility-maintained benchmarks and optimised builds.

In [ ]:
if workloads.empty or 'code_name' not in workloads.columns:
    print("No code name data available.")
else:
    code_counts = (
        workloads['code_name']
        .dropna()
        .loc[lambda s: s.str.strip() != '']
        .str.strip()
        .str.title()
        .value_counts()
        .head(15)
    )

    fig, ax = plt.subplots(figsize=(9, 5))
    bars = ax.barh(code_counts.index[::-1], code_counts.values[::-1], color=sns.color_palette('muted')[0])
    for bar, val in zip(bars, code_counts.values[::-1]):
        ax.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height() / 2,
                str(val), va='center', ha='left', fontsize=9)
    ax.set_xlabel('Number of groups using this code')
    ax.set_title('Top 15 Codes / Applications Used', fontsize=13)
    ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
    plt.tight_layout()
    plt.show()

---
## Chart 3 — Jobs Per Year by Code Category

Shows the volume of jobs submitted per year, broken down by workload category. Range strings are mapped to midpoint values for ordering (the axis shows the original range labels).

**What to look for:** Which categories generate high job volumes? High job counts may indicate throughput-sensitive workloads that benefit from a dedicated high-throughput queue.

In [ ]:
if workloads.empty or 'job_count_range' not in workloads.columns or 'categories' not in workloads.columns:
    print("No job count or category data available.")
else:
    df_j = workloads[['categories', 'job_count_range']].copy()
    df_j = df_j.dropna(subset=['job_count_range'])
    df_j = df_j.loc[df_j['job_count_range'].str.strip() != '']

    # Explode categories so each category gets its own row
    df_j['categories'] = df_j['categories'].apply(
        lambda x: [v.strip() for v in str(x).split(';') if v.strip()] if pd.notna(x) else []
    )
    df_j = df_j.explode('categories')
    df_j['category_label'] = df_j['categories'].map(
        lambda x: CATEGORY_LABELS.get(x, x) if pd.notna(x) else x
    )
    df_j['job_label'] = map_range_labels(df_j['job_count_range'], JOB_COUNT_LABELS)
    df_j['job_midpoint'] = df_j['job_count_range'].map(
        lambda x: JOB_COUNT_MIDPOINTS.get(str(x).strip(), 0)
    )

    pivot = df_j.groupby(['category_label', 'job_label'])['job_label'].count().unstack(fill_value=0)

    # Order columns by job count midpoint
    col_order = [
        lbl for lbl in JOB_COUNT_LABELS.values()
        if lbl in pivot.columns
    ]
    pivot = pivot[col_order]

    fig, ax = plt.subplots(figsize=(12, 5))
    pivot.plot(kind='bar', stacked=True, ax=ax, colormap='tab10')
    ax.set_xlabel('Workload category')
    ax.set_ylabel('Number of code entries')
    ax.set_title('Job Count Range Distribution by Workload Category', fontsize=13)
    ax.legend(title='Jobs per year', bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=8)
    plt.xticks(rotation=30, ha='right', fontsize=8)
    plt.tight_layout()
    plt.show()

---
## Chart 4 — Department Distribution

Shows how submitting groups are distributed across departments or schools. This is contextual information only — the committee does not assign QoS on the basis of department.

**What to look for:** Coverage across departments. Are there departments with no submissions that should be followed up?

In [ ]:
DEPT_LABELS = {
    'biological_sciences': 'Biological Sciences',
    'chemistry': 'Chemistry',
    'earth_environmental': 'Earth & Environmental',
    'economics': 'Economics',
    'humanities_social_sciences': 'Humanities & Social Sciences',
    'mathematics': 'Mathematics',
    'physics': 'Physics',
    'interdisciplinary': 'Interdisciplinary',
    'other': 'Other',
}

dept_col = None
for df_name, df in [('RespondentInfo', respondents), ('Submissions', dfs.get('Submissions', pd.DataFrame()))]:
    for col in ['department', 'A_department']:
        if col in df.columns:
            dept_col = (df, col)
            break
    if dept_col:
        break

if dept_col is None:
    print("No department data found in RespondentInfo or Submissions.")
else:
    src_df, col = dept_col
    dept_counts = (
        src_df[col]
        .dropna()
        .loc[lambda s: s.str.strip() != '']
        .map(lambda x: DEPT_LABELS.get(x.strip(), x.strip()))
        .value_counts()
    )

    fig, ax = plt.subplots(figsize=(7, 7))
    wedges, texts, autotexts = ax.pie(
        dept_counts.values,
        labels=dept_counts.index,
        autopct=lambda p: f'{int(round(p * dept_counts.sum() / 100))}',
        startangle=120,
        textprops={'fontsize': 9},
    )
    ax.set_title('Department Distribution of Submitting Groups', fontsize=13, pad=16)
    plt.tight_layout()
    plt.show()
    print(dept_counts.to_string())